#### 연습문제
1. data 폴더 안에 ratings_train.csv 파일을 로드
2. 결측치를 제외
3. id 컬럼 제외
4. 중복 데이터 제거
5. label이 0인 데이터 중 2000개를 추출, label이 1인 데이터 중 2000개를 추출
6. 5번 결과를 단순 행결합
7. train, test 데이터셋을 8 : 2의 비율로 나눠준다.
8. tokenizer는 Okt를 사용
9. 불필요한 품사는 제외 (사용할 품사 : 명사, 동사, 형용사, 부사, 파티클)
10. 글자수의 제한은 2자리부터 가능
11. tfidf를 사용하여 벡터화
    - min_df = 3
    - ngram_range=(1, 1)
12. 로지스틱 회귀 모델을 사용하여 벡터화한 데이터에서 학습
13. test데이터셋을 이용하여 검증 후 평가 지표 생성
14. 예측 결과, 원본의 데이터셋과 예측확률을 하나의 데이터프레임으로 생성
15. 결과물 제출 : 평가 지표, 14번의 결과에서 상위 5개

In [56]:
import pandas as pd
# import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer

In [57]:
#1. 
df=pd.read_csv('../../data/ratings_train.txt', sep='\t')
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [59]:
#2. 결측치 제거
df.dropna(inplace=True)

In [60]:
#3. id 컬럼 제외
df.drop('id',axis=1,inplace=True)

In [61]:
#4. 
df.drop_duplicates('document',inplace=True)

In [62]:
#5. 
#By 강사님
#label이 0인 데이터셋을 필터링하고 2000개의 데이터를 추출
df_0=df.loc[df['label'] == 0,]
df_0=df_0.iloc[:2000]

df_1=df.loc[df['label'] == 1].iloc[:2000]
#By Gemini
# df_label_0 = df[df['label'] == 0].sample(n=2000, random_state=42)
# df_label_1 = df[df['label'] == 1].sample(n=2000, random_state=42)

In [63]:
#6. 두 데이터를 합칩니다.
df2 = pd.concat([df_0, df_1], axis=0, ignore_index=True)
df2

,document,label
0,아 더빙.. 진짜 짜증나네요 목소리,0
1,너무재밓었다그래서보는것을추천한다,0
2,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
3,막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.,0
4,원작의 긴장감을 제대로 살려내지못했다.,0
...,...,...
3995,20년 전에 이런 영화가 있었다. 즐겁고 재밌고 감동도...,1
3996,되게 웃기다가 되게 울리는 영화,1
3997,잼난다..,1
3998,그냥 한번 보세요 ㅇ ㅇ ? ?,1


In [64]:
X=df2['document'].values
y=df2['label'].values

In [65]:
#7. 
X_train,X_test,y_train,y_test=train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [66]:
#8. 토크나이저 생성
okt=Okt()
#토큰화는 벡터화 작업에서 한번에 실행시키기 위해 함수 선언
def tokenize(text):
    #9. 품사 제한 : 명사, 동사, 형용사, 부사, 파티클
    allow_pos=['Noun','Verb','Adjective','Adverb','KoreanParticle']
    #10. 문자의 길이 제한 : 2보다 크거나 같다
    len_word=2
    result=[]
    for word, pos in okt.pos(text, norm=True, stem=True):
        #조건 1 : allow_pos에 포함되어 있으면
        #조건 2 : stop_word에 포함되어 있지 않다면
        #조건 3 : 길이가 len_word보다 크거나 같은 경우
        if (pos in allow_pos) & (len(word) >= len_word):
            result.append(word)
    return result

In [67]:
#11. 벡터화 class 생성
tfidf_vec=TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1, 1),
    min_df=2
)

In [ ]:
#X의 train과 test를 토큰화 + 벡터화
X_train_vec=tfidf_vec.fit_transform(X_train)
X_test_vec=tfidf_vec.transform(X_test)

In [69]:
#생성된 피쳐의 개수는 몇개?
print(len(tfidf_vec.get_feature_names_out()))

2239


In [70]:
#12. 
logistic=LogisticRegression(random_state=42)

#모델 학습
logistic.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [71]:
#13. 
pred=logistic.predict(X_test_vec)
print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       0.78      0.84      0.81       400
           1       0.83      0.76      0.79       400

    accuracy                           0.80       800
   macro avg       0.80      0.80      0.80       800
weighted avg       0.80      0.80      0.80       800



In [72]:
#14. 
#예측값의 확률
pred_proba=logistic.predict_proba(X_test_vec)
pred_proba

array([[0.11168225, 0.88831775],
       [0.20787891, 0.79212109],
       [0.94719288, 0.05280712],
       ...,
       [0.70219808, 0.29780192],
       [0.57889302, 0.42110698],
       [0.5154846 , 0.4845154 ]], shape=(800, 2))

In [ ]:
#X_test, pred, pred_proba 3개의 데이터를 반복문을 통해서 반복 실행 -> 2차원으로 새로운 데이터를 구성
data=[]
for review, value, proba in zip(X_test, pred, pred_proba):
    #value는 1이라면 '긍정', 0이라면 '부정'
    value= '긍정' if value == 1 else '부정'
    #proba : 둘중에 큰 값만 사용 -> 100을 곱한다. -> 소수점 3번째 자리에서 반올림 -> '%' 붙여준다.
    proba = round(max(proba) * 100,2)
    proba = str(proba) + '%'
    #review, value, proba 데이터를 하나의 리스트에 대입
    data.append([review, value, proba])
data

In [74]:
pd.DataFrame(data, columns=['review','Pred','Proba']).head(5)

,review,Pred,Proba
0,2시간동안 우느라 지칠정도.. 대사하나하나 울게 만드네요 최고,긍정,88.83%
1,스파이더맨을 가장 좋아하는 1人~ 스파이더맨 완전 사랑합니다. 영원했으면 좋겠네요!!,긍정,79.21%
2,쓰레기중의 쓰레기 영화 모든것이 쓰레기다.,부정,94.72%
3,이 편에 극장판 중에 제일 마음에 든다,긍정,60.73%
4,어렸을 때 봤을 때 영화 속 인어속 꼬리가 너무나 신비롭게 보였던.,긍정,61.7%


- 네이버 개발자 센터를 이용해서 데이터를 수집
- 수집된 데이터를 이용하여 감정 평가 예측
    1. 네이버 개발자 센터 접속
    2. 서비스 api 신청
    3, 서비스키를 이용해서 뉴스 데이터를 로드
    4. 데이터를 이용하여 감정 분석

In [75]:
import os
from dotenv import load_dotenv
import requests
import re

In [76]:
load_dotenv()

True

In [77]:
naver_id=os.getenv('naver_api_id')
naver_secret=os.getenv('naver_api_secret')

naver_id

'XB3kdjz0wjsD2Jz23FnO'

In [78]:
#네이버 api를 활용해서 news 제목들을 수집
url='https://openapi.naver.com/v1/search/news.json'

params={
    'query' : '왕사남',
    'display' : 30
}
headers={
    'X-Naver-Client-Id' : naver_id,
    'X-Naver-Client-Secret' : naver_secret,
}

res=requests.get(
    url=url,
    params=params,
    headers=headers,
)

print(res)
print(res.json)

<Response [200]>
<bound method Response.json of <Response [200]>>


1.res.json()에서 title 부분의 value를 추출하여 하나의 리스트로 생성
2. <b>,</b> 문자를 제거
3. 위에서 만들어둔 벡터화를 이용하여 벡터화 작업
4. 로지스틱 모델을 이용하여 예측
5. 예측 값과 확률을 데이터프레임으로 생성

In [79]:
new_titles=[]
for item in res.json()['items']:
    #item -> dict 형태 데이터가 대입
    # print(item['title'].replace('<b>','').replace('</b>',''))
    # break
    clean_title=re.sub(r'<[^>]*>','',item['title'])
    # print(clean_title)
    new_titles.append(clean_title)

In [80]:
#데이터를 벡터화
X_api=tfidf_vec.transform(new_titles)

In [81]:
X_api.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(30, 2239))

In [82]:
pred_api=logistic.predict(X_api)
proba_api=logistic.predict_proba(X_api)

In [83]:
data=[]
for title, value, proba in zip(new_titles, pred_api, proba_api):
    value='긍정' if value == 1 else '부정'
    proba=round(max(proba) * 100, 2)
    data.append(
        {
            'title' : title,
            'pred' : value,
            'pred_proba' : proba
        }
    )

df_api=pd.DataFrame(data)

In [84]:
df_api.sort_values('pred_proba', ascending=False).head(10)

,title,pred,pred_proba
17,"연상호 감독 '군체', 400만 향해 돌진 중-'왕사남' 넘을까?",부정,83.20
2,‘호프’ 찍은 해남에 ‘1970년대 거리’ 만든다,부정,75.00
4,단종 이어 안평대군…그들의 억울함은 이 시대 어떤 의미일까,긍정,69.20
24,"연간 2위 '군체', 350만명 최단기 기록…'왕사남'보다 2일 빨라",긍정,69.02
29,"‘군체’, ‘왕사남’보다 빠르다…10일 만에 300만 손익분기점 돌파",긍정,69.02
26,"‘군체’ 벌써 300만 돌파, ‘왕사남’보다 빠르다 [지금뉴스]",긍정,67.65
19,"'군체', 주말에만 97만명 봤다…흥행 장기 집권 기대",긍정,64.79
21,'군체'의 시대...감독·배우 드림팀+ K좀비의 또 다른 시작[MD이슈],부정,64.41
5,"밀양시, 얼음골 방문 인증 SNS 이벤트 6월 한달간 진행",부정,61.50
11,"밀양시, '반하다밀양 반값여행' 얼음골 SNS 이벤트 진행",부정,61.50


In [85]:
df_api

,title,pred,pred_proba
0,‘왕사남’ 흥행에 문경새재 ‘구름인파’…올봄 관람객 153만명 몰렸다,부정,55.20
1,"칸이 열고 관객이 채웠다... '군체', 357만 돌파→'왕사남' 이어 흥행 성...",부정,53.78
2,‘호프’ 찍은 해남에 ‘1970년대 거리’ 만든다,부정,75.00
3,'왕사남'이어 '살목지'도 흥행…4월 극장 매출 31.2%↑,부정,59.15
4,단종 이어 안평대군…그들의 억울함은 이 시대 어떤 의미일까,긍정,69.20
5,"밀양시, 얼음골 방문 인증 SNS 이벤트 6월 한달간 진행",부정,61.50
6,"'왕사남' 촬영지 문경새재, 올들어 153만명 찾았다",부정,56.12
7,'왕사남' 열풍에 찻사발축제 효과 '톡톡'…문경새재 방문객 153만 명 돌...,긍정,55.85
8,"반하다밀양 반값여행, 6월 얼음골 인증 이벤트로 열기 쭈~욱",부정,54.21
9,"'군체', 아시아 주요 지역 박스오피스 1위…글로벌 흥행 기대감",부정,57.62


#### 모델의 성능을 올려보자
- 실제 모델의 성능
    - 정확도 : 77%
- 모델의 성능을 올릴 수 있는 방법
    - 데이터의 양을 늘린다
    - 다른 전처리 방법 사용 (분석기, 벡터화)
    - 스케일러 이용 --> MaxAbs Scaler 사용
    - 벡터화 모델과 분류 모델의 파라미터 수정
    - 모델을 변경

In [86]:
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

In [87]:
ma_scaler=MaxAbsScaler()
cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe=Pipeline(
    [
        (
            'vec', tfidf_vec
        ),
        (
            'scaler', ma_scaler
        ),
        (
            'model', logistic
        )
    ]
)

In [91]:
#파라미터 조합 생성
params={
    'vec__min_df' : [2, 3],
    'vec__ngram_range' : [(1, 2), (1, 1)],
    'vec__max_features' : [None, 1000],
    'model__C' : [0.8, 0.9, 1.0],
}

In [92]:
grid=GridSearchCV(
    estimator=pipe,
    param_grid=params,
    cv=cv,
    verbose=1
)

In [ ]:
grid.fit(X, y)

In [95]:
grid.best_params_

{'model__C': 0.8,
 'vec__max_features': None,
 'vec__min_df': 2,
 'vec__ngram_range': (1, 2)}

In [96]:
pred_api=grid.predict(new_titles)
proba_api=grid.predict_proba(new_titles)

In [97]:
data=[]
for title, value, proba in zip(new_titles, pred_api, proba_api):
    value='긍정' if value == 1 else '부정'
    proba=round(max(proba) * 100, 2)
    data.append(
        {
            'title' : title,
            'pred' : value,
            'pred_proba' : proba
        }
    )

df_api=pd.DataFrame(data)

In [98]:
df_api.head(10)

,title,pred,pred_proba
0,‘왕사남’ 흥행에 문경새재 ‘구름인파’…올봄 관람객 153만명 몰렸다,긍정,57.60
1,"칸이 열고 관객이 채웠다... '군체', 357만 돌파→'왕사남' 이어 흥행 성...",부정,53.24
2,‘호프’ 찍은 해남에 ‘1970년대 거리’ 만든다,부정,66.56
3,'왕사남'이어 '살목지'도 흥행…4월 극장 매출 31.2%↑,부정,54.23
4,단종 이어 안평대군…그들의 억울함은 이 시대 어떤 의미일까,긍정,57.07
5,"밀양시, 얼음골 방문 인증 SNS 이벤트 6월 한달간 진행",부정,75.23
6,"'왕사남' 촬영지 문경새재, 올들어 153만명 찾았다",부정,60.33
7,'왕사남' 열풍에 찻사발축제 효과 '톡톡'…문경새재 방문객 153만 명 돌...,긍정,66.80
8,"반하다밀양 반값여행, 6월 얼음골 인증 이벤트로 열기 쭈~욱",긍정,52.86
9,"'군체', 아시아 주요 지역 박스오피스 1위…글로벌 흥행 기대감",부정,52.66


In [99]:
load_dotenv()

True

In [ ]:
youtube_api=os.getenv('youtube_api')
youtube_api

- 유튜브에서 특정 영상의 댓글을 로드
    - 구글 클라우드 콘솔에서 api를 신청
    - 신청이 된 api key와 영상의 id값이 필요
    - 라이브러리 설치
        - google-api-python-client

In [ ]:
# !pip install google-api-python-client

In [117]:
#영상의 id를 하나 복사
#유튜브 영상 url에서 가장 마지막에 v=ID
video_id='PhYJQNSQ4oI'

In [116]:
from googleapiclient.discovery import build

In [118]:
youtube=build('youtube', 'v3', developerKey=youtube_api)

In [126]:
request = youtube.commentThreads().list(
    part = 'snippet', 
    videoId = video_id, 
    maxResults = 10, 
    textFormat = 'plainText'
)

In [127]:
from pprint import pprint

In [ ]:
res=request.execute()

pprint(res['items'])

In [129]:
comment=[]

for item in res['items']:
    # pprint(item)
    # pprint(item['snippet'])
    # pprint(item['snippet']['topLevelComment'])
    # pprint(item['snippet']['topLevelComment']['snippet'])
    # pprint(item['snippet']['topLevelComment']['snippet']['textDisplay'])
    comment.append(item['snippet']['topLevelComment']['snippet']['textDisplay'])
    
comment

['간은 단단해지면 안 된다는 거 처음 안 기모찌금자는 개추',
 '3:09 왜 요서 장난 똥때리나로 들리지...ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ',
 '혹시라도 러끼+모찌+금자+겨울 이렇게 모이면 ....대환장파티네',
 '수문장이 뭔데 ㅋㅋㅋㅋㅋㅋㅋ',
 '그래서 언제 만나는거죠?',
 '참고로 술이 느는건 뇌가 망가지고 있다는 뜻....',
 '신데렐라도 12시에 들어가요 ~ㅋㅋㅋㅋㅋ',
 '나 아빠한테 존댓말쓰라고 혼났는ㄷ',
 '금버지님이 따라주시는 술... 너무 귀하네요.... 방바닥에 이마 대고 받아야 할 것 같음...',
 '김모찌에서 기모찌가 ㅋㅋㅋ\n이게 더 말하기 편하긴 하짘ㅋㅋㅋㅋ']

In [135]:
labels=[1, 0, 0, 1, 1, 1, 1, 0, 1, 1] # comment 긍부정 판단 후 일일이 설정
comment_df=pd.DataFrame(zip(comment, labels), columns=['review','label'])
comment_df

,review,label
0,간은 단단해지면 안 된다는 거 처음 안 기모찌금자는 개추,1
1,3:09 왜 요서 장난 똥때리나로 들리지...ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ,0
2,혹시라도 러끼+모찌+금자+겨울 이렇게 모이면 ....대환장파티네,0
3,수문장이 뭔데 ㅋㅋㅋㅋㅋㅋㅋ,1
4,그래서 언제 만나는거죠?,1
5,참고로 술이 느는건 뇌가 망가지고 있다는 뜻....,1
6,신데렐라도 12시에 들어가요 ~ㅋㅋㅋㅋㅋ,1
7,나 아빠한테 존댓말쓰라고 혼났는ㄷ,0
8,금버지님이 따라주시는 술... 너무 귀하네요.... 방바닥에 이마 대고 받아야 할 ...,1
9,김모찌에서 기모찌가 ㅋㅋㅋ\n이게 더 말하기 편하긴 하짘ㅋㅋㅋㅋ,1


In [136]:
pred=grid.predict(comment)
pred_proba=grid.predict_proba(comment)

pred

array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [137]:
print(classification_report(pred, labels))

              precision    recall  f1-score   support

           0       1.00      0.33      0.50         9
           1       0.14      1.00      0.25         1

    accuracy                           0.40        10
   macro avg       0.57      0.67      0.38        10
weighted avg       0.91      0.40      0.47        10



In [138]:
data=[]
for review, label, value, proba in zip(comment, labels, pred, pred_proba):
    label='긍정' if label == 1 else '부정'
    value='긍정' if value == 1 else '부정'
    proba=round(max(proba)*100,2)
    proba=str(proba)+'%'
    
    data.append(
        {
            'review':review,
            'origin':label,
            'pred':value,
            'proba':proba
        }
    )

df_youtube=pd.DataFrame(data)
df_youtube

,review,origin,pred,proba
0,간은 단단해지면 안 된다는 거 처음 안 기모찌금자는 개추,긍정,부정,59.75%
1,3:09 왜 요서 장난 똥때리나로 들리지...ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ,부정,부정,63.09%
2,혹시라도 러끼+모찌+금자+겨울 이렇게 모이면 ....대환장파티네,부정,부정,53.83%
3,수문장이 뭔데 ㅋㅋㅋㅋㅋㅋㅋ,긍정,부정,54.5%
4,그래서 언제 만나는거죠?,긍정,긍정,69.43%
5,참고로 술이 느는건 뇌가 망가지고 있다는 뜻....,긍정,부정,76.16%
6,신데렐라도 12시에 들어가요 ~ㅋㅋㅋㅋㅋ,긍정,부정,54.06%
7,나 아빠한테 존댓말쓰라고 혼났는ㄷ,부정,부정,65.99%
8,금버지님이 따라주시는 술... 너무 귀하네요.... 방바닥에 이마 대고 받아야 할 ...,긍정,부정,75.46%
9,김모찌에서 기모찌가 ㅋㅋㅋ\n이게 더 말하기 편하긴 하짘ㅋㅋㅋㅋ,긍정,부정,64.95%
